In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch7. 라이브러리를 이용한 시계열 데이터 분석</font>**
- statsmodels : 통계분석을 하기 위한 라이브러리 (회귀분석, 시계열분석, 가설검정,기술통계)
    * 주기적인 데이터의 트렌드 추이
- Prophet : 계정설 추세, 휴일효과 자동으로 모델링

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
# 한글설정
plt.rc('font', family='Malgun Gothic') # 윈도우즈 한글 깨짐 해결
plt.rc('axes', unicode_minus=False) # 축의 - 깨짐 해결

In [8]:
# data.go.kr / 에어코리아(AirKorea) 다운로드
df = pd.read_csv('data/일별평균대기오염도_2022(에어코리아).csv', encoding='cp949')
df.tail()

,측정일시,측정소명,이산화질소농도(ppm),오존농도(ppm),일산화탄소농도(ppm),아황산가스농도(ppm),미세먼지농도(㎍/㎥),초미세먼지농도(㎍/㎥)
18245,20221231,구로구,0.037,0.009,0.5,0.004,43.0,29.0
18246,20221231,광진구,0.026,0.005,0.8,0.003,44.0,34.0
18247,20221231,관악산,0.008,0.038,0.3,0.005,29.0,18.0
18248,20221231,관악구,0.045,0.009,0.7,0.003,42.0,28.0
18249,20221231,공항대로,0.042,0.007,0.7,0.004,41.0,31.0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18250 entries, 0 to 18249
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   측정일시          18250 non-null  int64  
 1   측정소명          18250 non-null  object 
 2   이산화질소농도(ppm)  18172 non-null  float64
 3   오존농도(ppm)     18176 non-null  float64
 4   일산화탄소농도(ppm)  18174 non-null  float64
 5   아황산가스농도(ppm)  18176 non-null  float64
 6   미세먼지농도(㎍/㎥)   18115 non-null  float64
 7   초미세먼지농도(㎍/㎥)  18122 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 1.1+ MB


In [18]:
# 결측치 있는 데이터
df[df['이산화질소농도(ppm)'].isna()]
df[df.isna().any(axis=1)] # 결측치가 한열이라도 있는 경우
df.loc[df.isna().any(axis=1),'측정소명'].value_counts() # 결측치가 한열이라도 있는 측정소명
# len(df.loc[df.isna().any(axis=1),'측정소명'].value_counts() ) 21개 측정소가 결측치가 있음

남산        43
동대문구      24
관악산       13
한강대로      12
시흥대로      11
북한산       11
항동         9
행주         5
마포아트센터     4
강변북로       4
서초구        3
성북구        2
송파구        2
동작대로       2
세곡         2
도산대로       2
서대문구       2
관악구        2
올림픽공원      1
은평구        1
동작구        1
Name: 측정소명, dtype: int64

In [21]:
# 서울시 측정소명들
print('측정소들 : ',df['측정소명'].unique())
print('측정소 갯수 : ',df['측정소명'].nunique())

측정소들 :  ['강남구' '홍릉로' '행주' '항동' '한강대로' '청계천로' '천호대로' '중랑구' '중구' '종로구' '종로' '정릉로'
 '자연사박물관' '은평구' '용산구' '올림픽공원' '영등포로' '영등포구' '양천구' '신촌로' '시흥대로' '송파구' '세곡'
 '성북구' '성동구' '화랑로' '서초구' '서울숲' '서대문구' '북한산' '마포아트센터' '마포구' '동작대로' '동작구'
 '동대문구' '도산대로' '도봉구' '노원구' '남산' '금천구' '구로구' '광진구' '관악산' '관악구' '공항대로' '강서구'
 '강북구' '강변북로' '강동구' '강남대로']
측정소 갯수 :  50


In [33]:
# int형인 측정일시 컬럼을 str로
df['측정일시'] = df['측정일시'].astype(str)
df['측정일시'] = pd.to_datetime(df['측정일시'])
# df['측정일시'] = df['측정일시'].astype('datetime64[ns]')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18250 entries, 0 to 18249
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   측정일시          18250 non-null  datetime64[ns]
 1   측정소명          18250 non-null  object        
 2   이산화질소농도(ppm)  18172 non-null  float64       
 3   오존농도(ppm)     18176 non-null  float64       
 4   일산화탄소농도(ppm)  18174 non-null  float64       
 5   아황산가스농도(ppm)  18176 non-null  float64       
 6   미세먼지농도(㎍/㎥)   18115 non-null  float64       
 7   초미세먼지농도(㎍/㎥)  18122 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 1.1+ MB


In [39]:
# 50개 측정소명을 매일 기록(그 중 결측치가 있을 수 있음)
df['측정소명'].value_counts().sort_index().index

Index(['강남구', '강남대로', '강동구', '강변북로', '강북구', '강서구', '공항대로', '관악구', '관악산', '광진구',
       '구로구', '금천구', '남산', '노원구', '도봉구', '도산대로', '동대문구', '동작구', '동작대로', '마포구',
       '마포아트센터', '북한산', '서대문구', '서울숲', '서초구', '성동구', '성북구', '세곡', '송파구',
       '시흥대로', '신촌로', '양천구', '영등포구', '영등포로', '올림픽공원', '용산구', '은평구', '자연사박물관',
       '정릉로', '종로', '종로구', '중구', '중랑구', '천호대로', '청계천로', '한강대로', '항동', '행주',
       '홍릉로', '화랑로'],
      dtype='object')

In [45]:
# 미세먼지와 초미세먼지 컬럼에 결측치가 없는 측정소명
df[df['측정소명']=='공항대로'].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 365 entries, 44 to 18249
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   측정일시          365 non-null    datetime64[ns]
 1   측정소명          365 non-null    object        
 2   이산화질소농도(ppm)  365 non-null    float64       
 3   오존농도(ppm)     365 non-null    float64       
 4   일산화탄소농도(ppm)  365 non-null    float64       
 5   아황산가스농도(ppm)  365 non-null    float64       
 6   미세먼지농도(㎍/㎥)   365 non-null    float64       
 7   초미세먼지농도(㎍/㎥)  365 non-null    float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 25.7+ KB


In [49]:
loc_name = '관악구' # 관악구는 결측치가 있는 측정소명
df_flt = df[df['측정소명']==loc_name]
df_flt.head()

,측정일시,측정소명,이산화질소농도(ppm),오존농도(ppm),일산화탄소농도(ppm),아황산가스농도(ppm),미세먼지농도(㎍/㎥),초미세먼지농도(㎍/㎥)
43,2022-01-01,관악구,0.037,0.011,0.6,0.003,24.0,13.0
92,2022-01-02,관악구,0.034,0.013,0.6,0.003,31.0,22.0
142,2022-01-03,관악구,0.040,0.009,0.6,0.003,23.0,13.0
195,2022-01-04,관악구,0.026,0.018,0.5,0.003,30.0,18.0
243,2022-01-05,관악구,0.048,0.005,0.8,0.004,40.0,27.0


In [ ]:
# statsmode를 통한 추이 탐색